In [0]:
%python
from pyspark.sql import SparkSession

GOLD_DIR = "/Volumes/workspace/default/my_volume/drone_pipeline/data/gold"
TABLES = ["drone_kpis", "avg_delivery_time", "battery_efficiency", "high_risk_zones"]
for table in TABLES:
        df = spark.read.parquet(f"{GOLD_DIR}/{table}")
        df.createOrReplaceTempView(table)

In [0]:
SELECT drone_id, model, total_flights, avg_success_rate, failure_rate
FROM drone_kpis
ORDER BY failure_rate DESC
LIMIT 5;

drone_id,model,total_flights,avg_success_rate,failure_rate
2,CargoWing,29,68.97,31.03
7,CargoWing,15,73.33,26.67
15,AeroMule,21,76.19,23.81
12,SkyHawk200,22,77.27,22.73
14,SwiftDrone3,29,79.31,20.69


In [0]:
-- 2. Fleet-wide average success rate
SELECT ROUND(AVG(avg_success_rate), 2) AS fleet_avg_success_rate
FROM drone_kpis;


fleet_avg_success_rate
85.07


In [0]:
-- 3. Slowest drones by average delivery time
SELECT drone_id, avg_delivery_time_minutes
FROM avg_delivery_time
ORDER BY avg_delivery_time_minutes DESC
LIMIT 5;


drone_id,avg_delivery_time_minutes
16,58.63
11,56.26
20,53.58
13,52.92
9,51.33


In [0]:
-- 4. Most battery-efficient drone models (join back to drones for model name)
SELECT d.model, ROUND(AVG(b.avg_battery_efficiency), 3) AS model_avg_efficiency
FROM battery_efficiency b
JOIN drone_kpis d ON b.drone_id = d.drone_id
GROUP BY d.model
ORDER BY model_avg_efficiency DESC;

model,model_avg_efficiency
SkyHawk200,0.47
FalconX1,0.458
CargoWing,0.45
SwiftDrone3,0.448
AeroMule,0.444


In [0]:
-- 5. Top 5 highest-risk delivery zones
SELECT destination, failure_count
FROM high_risk_zones
ORDER BY failure_count DESC
LIMIT 5;



destination,failure_count
Downtown,14
Northgate,12
Harbor View,11
Hillcrest,10
Riverside,10


In [0]:
-- 6. Drones that are both slow AND unreliable (needs attention first)
SELECT k.drone_id, k.model, k.failure_rate, t.avg_delivery_time_minutes
FROM drone_kpis k
JOIN avg_delivery_time t ON k.drone_id = t.drone_id
WHERE k.failure_rate > 20 AND t.avg_delivery_time_minutes > (
    SELECT AVG(avg_delivery_time_minutes) FROM avg_delivery_time
)
ORDER BY k.failure_rate DESC;

drone_id,model,failure_rate,avg_delivery_time_minutes
7,CargoWing,26.67,48.62
15,AeroMule,23.81,51.13
12,SkyHawk200,22.73,48.98
14,SwiftDrone3,20.69,50.11
